# Data Ingestion — DataCo Smart Supply Chain Dataset

This notebook handles the first stage of the supply chain analytics pipeline: pulling the raw dataset from Kaggle, performing an initial inspection of its structure, and persisting a local copy for downstream transformation.

**Pipeline stage:** Raw data acquisition  
**Input:** DataCo Smart Supply Chain dataset (Kaggle)  
**Output:** `../data/raw/DataCoSupplyChain.csv`

## 1. Import Libraries and Download Dataset

The dataset is pulled directly from Kaggle using the `kagglehub` package, which authenticates via credentials loaded from a local `.env` file. The CSV is read with `latin-1` encoding to handle non-ASCII characters present in the source data.

In [43]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv() 


# Load the latest version
path = kagglehub.dataset_download("shashwatwork/dataco-smart-supply-chain-for-big-data-analysis")
df = pd.read_csv(f"{path}/DataCoSupplyChainDataset.csv", encoding="latin-1")

## 2. Initial Data Inspection

Before any cleaning or modeling, the dataset is profiled to understand its shape, column structure, and data types. This step informs which fields will require type conversion or null handling in the transformation stage.

In [44]:
df.shape
df.columns
df.dtypes

Type                                 str
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                      str
Late_delivery_risk                 int64
Category Id                        int64
Category Name                        str
Customer City                        str
Customer Country                     str
Customer Email                       str
Customer Fname                       str
Customer Id                        int64
Customer Lname                       str
Customer Password                    str
Customer Segment                     str
Customer State                       str
Customer Street                      str
Customer Zipcode                 float64
Department Id                      int64
Department Name                      str
Latitude                         float64
Longitude                        float64
Market          

### Null Value Audit

Counting nulls per column ensures that critical fields used as primary keys or core metrics, such as `Order Id`, `Customer Id`, and `Order Item Quantity` are complete. Columns with substantial nulls are flagged for downstream handling.

In [45]:
df.isnull().sum()

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


### Sample Records

A preview of the first rows provides a quick check on the data contents and confirms the encoding was applied correctly.

In [46]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## 3. Persist Raw Data

The raw dataset is written to a dedicated `data/raw/` directory. Keeping a local copy of the unmodified source data is a standard practice in data engineering — it allows the transformation pipeline to be re-run without re-downloading and provides an audit trail of what was originally ingested.

In [47]:
os.makedirs("../data/raw", exist_ok=True)

df.to_csv("../data/raw/DataCoSupplyChain.csv")

## Next Steps

The raw dataset is now ready for the cleaning and modeling stage in `data-transform.ipynb`, where it will be restructured into a star schema (fact and dimension tables) for consumption by Power BI.